# 03 Eval + Publish Colab
Loads Stage A/B/C artifacts, rebuilds plots/tables, and optionally uploads to HF.

In [ ]:
import os
REPO_URL='https://github.com/Chirag0096/ShiftLog-Gym.git'
REPO_DIR='ShiftLog-Gym'
if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}
!pip -q install -e . pandas matplotlib seaborn huggingface_hub wandb


In [ ]:
import os
from getpass import getpass
from train.colab_eval_publish import EvalPublishPipeline

wandb_key=os.environ.get('WANDB_API_KEY','').strip() or getpass('Enter WANDB_API_KEY (blank to skip W&B): ').strip()
hf_token=os.environ.get('HF_TOKEN','').strip() or getpass('Enter HF_TOKEN (blank to skip HF login): ').strip()

pipe=EvalPublishPipeline()
pipe.authenticate(hf_token=hf_token, wandb_key=wandb_key)
missing=pipe.load_curves()
print('Missing curves:' if missing else 'All curves present', missing)
plot_paths=pipe.plot_curves()
print('Plots:', [str(p) for p in plot_paths])
summary_path, comparison_path = pipe.write_tables()
print('Tables:', summary_path, comparison_path)


In [ ]:
PUBLISH_TO_HF=False
HF_MODEL_REPO=os.environ.get('HF_MODEL_REPO','Chirag0096/shiftlog-gym-qwen2.5-1.5b-memory-policy')
if PUBLISH_TO_HF:
    pipe.upload_bundle(HF_MODEL_REPO)
    print('Uploaded eval bundle to', HF_MODEL_REPO)
else:
    print('Set PUBLISH_TO_HF=True to upload eval bundle.')
